# Session 1A — The Passive LLM (The Baseline)

**GOAL:** Prove that a standard LLM is ONLY a text predictor.  
It can read and summarize emails, but it CANNOT actually schedule a meeting or take any real-world action.

In [ ]:
import os, sys, warnings, logging

# Suppress noisy logs
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Add project root to path so 'utils' package is importable
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

try:
    import utils.dns_patch
except Exception:
    pass

import google.generativeai as genai
from utils.gmail_utils import fetch_recent_emails
from utils.tools import format_emails

genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))
print('Setup complete.')

### Step 1: Fetch Real Emails from Gmail

In [ ]:
print('Fetching your 5 most recent emails ...')
emails = fetch_recent_emails(limit=5)

if not emails:
    raise RuntimeError('No emails found. Make sure credentials.json is set up and you have run the OAuth flow.')

print(f'Found {len(emails)} emails.')

### Step 2: Format Emails & Send to the LLM

We ask the LLM to categorize emails **and** schedule a meeting.  
Watch what happens...

In [ ]:
email_text = format_emails(emails, include_date=False)

prompt = f"""Here are my recent emails:
{email_text}

Please do the following:
1. Categorize each email (urgent / meeting / task / informational).
2. Identify any meeting requests and extract the proposed time.
3. **Schedule the meeting on my Google Calendar right now.**
4. Give me a summary of what you did.
"""

print('Sending emails to Gemini LLM ...')
model = genai.GenerativeModel('gemini-flash-latest')
response = model.generate_content(prompt)

print('\nLLM Response:')
print('-' * 50)
try:
    print(response.text)
except ValueError:
    # Gemini sometimes tries to generate a function call even without tools configured.
    # This PROVES our point: the LLM is desperately trying to act but has no tools!
    print('The LLM tried to generate a function call instead of text!')
    print('It literally WANTS to schedule the meeting but CANNOT.')
    print(f'Raw response parts: {response.candidates[0].content.parts}')
    print(f'Finish reason: {response.candidates[0].finish_reason}')
print('-' * 50)

### The Lesson

The LLM either **says** it scheduled the meeting (but nothing happened on your calendar), or it **tried to generate a function call** (and crashed because it has no tools).

Either way: **a standard LLM is a TEXT PREDICTOR.** It has NO ability to call APIs or execute code.

👉 **THIS is why we need AGENTS** (see Demo 1B).